In [2]:
!pip install kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kornia]2m1/2 [kornia]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Cell 1: Setup, Configuration & A100 Optimizations
# !pip install kornia kaggle diffusers accelerate

import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import math

import torch
import torch.nn as nn
import torch.fft
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
import kornia

# --- A100 SPECIFIC OPTIMIZATIONS ---
# Forces cuDNN to benchmark and select the fastest convolution algorithms for your static 456x456 shapes
torch.backends.cudnn.benchmark = True 
# Utilize TensorFloat-32 (TF32) on A100 Ampere architecture for massive speedups in matrix math
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")

# --- HYPERPARAMETERS ---
IMAGE_SIZE = 456 
BATCH_SIZE = 64      # A100 40GB VRAM comfortably handles this without accumulation
MAX_EPOCHS = 35      
PATIENCE = 5         
LR = 1e-4
VAL_SPLIT = 0.1

DATA_DIR = "./data" # Update this to your local NVMe or Kaggle input path

In [ ]:
# Cell 2: Distortion Library (Original & Complete)

# ==========================================
# CONSOLIDATED DISTORTION LIBRARY (ORIGINAL)
# ==========================================

# --- PART 1: UTILS & MATH ---
def sign(x: float) -> int: return 1 if x >= 0 else -1

def mapmm(x: torch.Tensor) -> torch.Tensor:
    minx, maxx = torch.min(x), torch.max(x)
    return (x - minx) / (maxx - minx) if minx < maxx else x

def fspecial(filter_type: str, p2: Union[int, Tuple[int, int]], p3: Union[int, float] = None) -> np.ndarray:
    if filter_type == 'gaussian':
        m, n = [(ss - 1.) / 2. for ss in p2]
        y, x = np.ogrid[-m:m + 1, -n:n + 1]
        h = np.exp(-(x * x + y * y) / (2. * p3 * p3))
        h[h < np.finfo(h.dtype).eps * h.max()] = 0
        sumh = h.sum()
        if sumh != 0: h /= sumh
        return h
    elif filter_type == 'disk':
        rad = p2; crad = math.ceil(rad - 0.5)
        x, y = np.ogrid[-rad: rad + 1, -rad: rad + 1]
        y = np.tile(y.transpose(), y.shape[1]); x = np.tile(x, x.shape[0]).transpose()
        y = np.abs(y); x = np.abs(x)
        maxxy = np.maximum(x, y); minxy = np.minimum(x, y)
        r1 = (rad ** 2 - (maxxy + 0.5) ** 2); r2 = (rad ** 2 - (minxy - 0.5) ** 2)
        warn_m1 = r1 ** 0.5 if (r1 > 0).all() else 0
        warn_m2 = r2 ** 0.5 if (r2 > 0).all() else 0
        m1 = (rad ** 2 < (maxxy + 0.5) ** 2 + (minxy - 0.5) ** 2) * (minxy - 0.5) + \
             (rad ** 2 >= (maxxy + 0.5) ** 2 + (minxy - 0.5) ** 2) * warn_m1
        m2 = (rad ** 2 > (maxxy - 0.5) ** 2 + (minxy + 0.5) ** 2) * (minxy + 0.5) + \
             (rad ** 2 <= (maxxy - 0.5) ** 2 + (minxy + 0.5) ** 2) * warn_m2
        sgrid = (rad ** 2 * (0.5 * (np.arcsin(m2 / rad) - np.arcsin(m1 / rad)) +
                             0.25 * (np.sin(2 * np.arcsin(m2 / rad)) - np.sin(2 * np.arcsin(m1 / rad)))) - (
                         maxxy - 0.5) * (m2 - m1) + (m1 - minxy + 0.5)) * np.logical_or(
            np.logical_and((rad ** 2 < (maxxy + 0.5) ** 2 + (minxy + 0.5) ** 2),
                           (rad ** 2 > (maxxy - 0.5) ** 2 + (minxy - 0.5) ** 2)),
            np.logical_and(np.logical_and(minxy == 0, maxxy - 0.5 < rad), maxxy + 0.5 >= rad))
        sgrid = sgrid + ((maxxy + 0.5) ** 2 + (minxy + 0.5) ** 2 < rad ** 2)
        sgrid[crad, crad] = np.minimum(math.pi * rad ** 2, math.pi / 2)
        if (crad > 0) and (rad > crad - 0.5) and (rad ** 2 < (crad - 0.5) ** 2 + 0.25):
            m1 = np.sqrt(rad ** 2 - (crad - 0.5) ** 2); m1n = m1 / rad
            sg0 = 2 * (rad ** 2 * (0.5 * np.arcsin(m1n) + 0.25 * np.sin(2 * np.arcsin(m1n))) - m1 * (crad - 0.5))
            sgrid[2 * crad, crad] = sg0; sgrid[crad, 2 * crad] = sg0
            sgrid[crad, 0] = sg0; sgrid[0, crad] = sg0
            sgrid[2 * crad, crad] -= sg0; sgrid[crad, 2 * crad] -= sg0
            sgrid[crad, 2] -= sg0; sgrid[2, crad] = sgrid[2, crad + 1] - sg0
        sgrid[crad, crad] = np.minimum(sgrid[crad, crad], 1)
        h = sgrid / np.sum(sgrid)
        return h
    else: raise NotImplementedError(f"Filter type {filter_type} not implemented")

def filter2D(img: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    img = img.float()
    k1 = kernel.size(-2); k2 = kernel.size(-1)
    b, c, h, w = img.size()
    if k1 % 2 == 1 or k2 % 2 == 1:
        img = F.pad(img, (k2 // 2, k2 // 2, k1 // 2, k1 // 2), mode='replicate')
    else: raise ValueError('Wrong kernel size')
    ph, pw = img.size()[-2:]
    if kernel.size(0) == 1:
        img = img.view(b * c, 1, ph, pw)
        kernel = kernel.view(1, 1, k1, k2)
        return F.conv2d(img, kernel, padding=0).view(b, c, h, w)
    else:
        img = img.view(1, b * c, ph, pw)
        kernel = kernel.view(b, 1, k1, k2).repeat(1, c, 1, 1).view(b * c, 1, k1, k2)
        return F.conv2d(img, kernel, groups=b * c).view(b, c, h, w)

def spline(x: np.ndarray, y: np.ndarray) -> tuple:
    n = x.shape[0]; dd = 1; dx = np.diff(x); divdif = np.diff(y) / dx
    if n == 3:
        y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]
        dlk = y[[2, 1, 0]].shape[0]; l = x[[0, 2]].shape[0] - 1; dl = np.prod(dd) * l
        k = np.fix(dlk / dl + 100 * 2.2204e-16); pp = (x[[0, 2]], y[[2, 1, 0]], l, int(k), dd)
    elif n > 3:
        b = np.zeros(n); b[1:n - 1] = 3 * (dx[1:n] * divdif[0:n - 2] + dx[0:n - 2] * divdif[1:n])
        x31 = x[2] - x[0]; xn = x[n - 1] - x[n - 3]
        b[0] = ((dx[0] + 2 * x31) * dx[1] * divdif[0] + dx[0] ** 2 * divdif[1]) / x31
        b[n - 1] = (dx[n - 2] ** 2 * divdif[n - 3] + (2 * xn + dx[n - 2]) * dx[n - 3] * divdif[n - 2]) / xn
        dxt = dx.T; c = np.zeros((3, 5))
        c[0, :] = [x31] + list(dxt[0:n - 2]) + [0]
        c[1, :] = [dxt[1]] + list(2 * (dxt[1:n - 1] + dxt[0:n - 2])) + [dxt[n - 3]]
        c[2, :] = [0] + list(dxt[1:n - 1]) + [xn]
        c = scipy.sparse.dia_matrix((c, [-1, 0, 1]), shape=(5, 5))
        c = scipy.sparse.csc_matrix(c); ic = scipy.sparse.linalg.inv(c); s = b * ic
        dxd = dx; dzzdx = (divdif - s[0:n - 1]) / dxd; dzdxdx = (s[1:n] - divdif) / dxd
        coefs = np.vstack(((dzdxdx - dzzdx) / dxd, 2 * dzzdx - dzdxdx, s[0:n - 1], y[0:n - 1])).T
        pp = (x, coefs, x.shape[0], x.shape[0], 1)
    else: raise ValueError('x.shape[0] must be >= 3')
    return pp

def histc(x: torch.Tensor, binranges: torch.Tensor) -> torch.Tensor:
    indices = torch.bucketize(x, binranges)
    return torch.remainder(indices, len(binranges)) - 1

def ppval(pp: tuple, xx: torch.Tensor) -> torch.Tensor:
    lx = torch.numel(xx); xs = xx.reshape(1, lx)
    b, c, l, k, dd = pp
    b = torch.as_tensor(b, device=xx.device)
    ranges = b.clone(); ranges[0] = -torch.inf; ranges[-1] = torch.inf
    index = histc(xs, ranges); xs = xs - b[index]
    c = torch.as_tensor(c, device=xx.device)
    if len(c.shape) == 1:
        v = c[0]
        for i in range(1, k): v = xs * v + c[i]
    else:
        v = c[index, 0]
        for i in range(1, k - 1): v = xs * v + c[index, i]
    v = v.view(xx.shape)
    return v

def curves(xx: torch.Tensor, coef: float) -> torch.Tensor:
    if isinstance(coef, list): coef = [[0.3, 0.5, 0.7], [coef[0], 0.5, coef[1]]]
    else: coef = [[0.5], [coef]]
    x = np.array([0] + [p for p in coef[0]] + [1])
    y = np.array([0] + [p for p in coef[1]] + [1])
    cs = spline(x, y); yy = ppval(cs, xx); yy = torch.clamp(yy, 0, 1)
    return yy

def bilinear_interpolate_torch(im: torch.Tensor, x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    dtype_long = torch.LongTensor
    x0 = torch.floor(x).type(dtype_long).to(im.device); x1 = x0 + 1
    y0 = torch.floor(y).type(dtype_long).to(im.device); y1 = y0 + 1
    x0 = torch.clamp(x0, 0, im.shape[1] - 1); x1 = torch.clamp(x1, 0, im.shape[1] - 1)
    y0 = torch.clamp(y0, 0, im.shape[0] - 1); y1 = torch.clamp(y1, 0, im.shape[0] - 1)
    Ia = im[y0, x0]; Ib = im[y1, x0]; Ic = im[y0, x1]; Id = im[y1, x1]
    R1 = Ia * (x1 - x) / (x1 - x0 + eps) + Ic * (x - x0) / (x1 - x0 + eps)
    R2 = Ib * (x1 - x) / (x1 - x0 + eps) + Id * (x - x0) / (x1 - x0 + eps)
    P = R1 * (y1 - y) / (y1 - y0 + eps) + R2 * (y - y0) / (y1 - y0 + eps)
    return P

def imscatter(x: torch.Tensor, amount: float, iterations=1) -> torch.Tensor:
    y = x
    for i in range(iterations):
        shiftmap = torch.randn((2, x.shape[1], x.shape[2]), device=x.device) * amount
        sy = shiftmap[0, :, :]; sx = shiftmap[1, :, :]
        m_sx = torch.ceil(torch.abs(torch.max(sx))).to(torch.int32)
        m_sy = torch.ceil(torch.abs(torch.max(sy))).to(torch.int32)
        y = F.pad(y, (m_sy, m_sy), mode='replicate')
        y = F.pad(y.transpose(2, 1), (m_sx, m_sx), mode='replicate').transpose(2, 1)
        sy = F.pad(sy, (m_sy, m_sy), mode='replicate')
        sy = F.pad(sy.transpose(1, 0), (m_sx, m_sx), mode='replicate').transpose(1, 0)
        sx = F.pad(sx, (m_sy, m_sy), mode='replicate')
        sx = F.pad(sx.transpose(1, 0), (m_sx, m_sx), mode='replicate').transpose(1, 0)
        xx, yy = torch.as_tensor(np.mgrid[0:y.shape[1], 0:y.shape[2]], device=x.device)
        z = torch.zeros_like(y)
        bx = (xx - sx); by = (yy - sy)
        for i in range(3):
            j = bilinear_interpolate_torch(y[i, ...], by, bx)
            z[i, :, :] = j
        y = z[:, m_sy:m_sy + x.shape[1], m_sx:m_sx + x.shape[2]]
    return y

# --- PART 2: DISTORTION FUNCTIONS ---
def gaussian_blur(x: torch.Tensor, blur_sigma: int = 0.1) -> torch.Tensor:
    fs = 2 * math.ceil(2 * blur_sigma) + 1
    h = fspecial('gaussian', (fs, fs), blur_sigma)
    h = torch.from_numpy(h).float()
    if len(x.shape) == 3: x = x.unsqueeze(0)
    y = filter2D(x, h.unsqueeze(0)).squeeze(0)
    return y

def lens_blur(x: torch.Tensor, radius: int) -> torch.Tensor:
    h = fspecial('disk', radius)
    h = torch.from_numpy(h).float()
    if len(x.shape) == 3: x = x.unsqueeze(0)
    y = filter2D(x, h.unsqueeze(0)).squeeze(0)
    return y

def color_saturation(x: torch.Tensor, factor: int) -> torch.Tensor:
    x = x[[2, 1, 0], ...]
    hsv = kornia.color.rgb_to_hsv(x)
    hsv[1, ...] *= factor
    y = kornia.color.hsv_to_rgb(hsv)
    return y[[2, 1, 0], ...]

def color_shift(x: torch.Tensor, amount: int) -> torch.Tensor:
    def perc(x, perc):
        xs = torch.sort(x)
        i = len(xs) * perc / 100.
        i = max(min(i, len(xs)), 1)
        v = xs[round(i - 1)]
        return v
    gray = kornia.color.rgb_to_grayscale(x)
    gradxy = kornia.filters.spatial_gradient(gray.unsqueeze(0), 'diff')
    e = torch.sum(gradxy ** 2, 2) ** 0.5
    fs = 2 * math.ceil(2 * 4) + 1
    h = fspecial('gaussian', (fs, fs), 4)
    h = torch.from_numpy(h).float()
    e = filter2D(e, h.unsqueeze(0))
    mine, maxe = torch.min(e), torch.max(e)
    if mine < maxe: e = (e - mine) / (maxe - mine)
    percdev = [1, 1]
    valuehi = perc(e, 100 - percdev[1]); valuelo = 1 - perc(1 - e, 100 - percdev[0])
    e = torch.max(torch.min(e, valuehi), valuelo)
    channel = 1
    g = x[channel, :, :]
    a = np.random.random((1, 2))
    amount_shift = np.round(a / (np.sum(a ** 2) ** 0.5) * amount)[0].astype(int)
    y = F.pad(g, (amount_shift[0], amount_shift[0]), mode='replicate')
    y = F.pad(y.transpose(1, 0), (amount_shift[1], amount_shift[1]), mode='replicate').transpose(1, 0)
    y = torch.roll(y, (amount_shift[0], amount_shift[1]), dims=(0, 1))
    if amount_shift[1] != 0: y = y[amount_shift[1]:-amount_shift[1], ...]
    if amount_shift[0] != 0: y = y[..., amount_shift[0]:-amount_shift[0]]
    yblend = y * e + x[channel, ...] * (1 - e)
    x[channel, ...] = yblend
    return x

def jpeg(x: torch.Tensor, quality: int) -> torch.Tensor:
    x_loc = x.clone() * 255.
    x_loc = x_loc.clamp(0, 255)
    y = encode_jpeg(x_loc.byte().cpu(), quality=quality)
    y = (decode_jpeg(y) / 255.).to(torch.float32)
    return y

def white_noise(x: torch.Tensor, var: float, clip: bool = True, rounds: bool = False) -> torch.Tensor:
    noise = torch.randn(*x.size(), dtype=x.dtype) * math.sqrt(var)
    y = x + noise
    if clip and rounds: y = torch.clip((y * 255.0).round(), 0, 255) / 255.
    elif clip: y = torch.clip(y, 0, 1)
    elif rounds: y = (y * 255.0).round() / 255.
    return y

def impulse_noise(x: torch.Tensor, d: float, s_vs_p: float = 0.5) -> torch.Tensor:
    num_sp = int(d * x.shape[0] * x.shape[1] * x.shape[2])
    coords = np.concatenate((np.random.randint(0, 3, (num_sp, 1)),
                             np.random.randint(0, x.shape[1], (num_sp, 1)),
                             np.random.randint(0, x.shape[2], (num_sp, 1))), 1)
    num_salt = int(s_vs_p * num_sp)
    coords_salt = coords[:num_salt].transpose(1, 0)
    coords_pepper = coords[num_salt:].transpose(1, 0)
    x[tuple(coords_salt)] = 1; x[tuple(coords_pepper)] = 0
    return x

def brighten(x: torch.Tensor, amount: float) -> torch.Tensor:
    x = x[[2, 1, 0]]
    lab = kornia.color.rgb_to_lab(x)
    l = lab[0, ...] / 100.
    l_ = curves(l, 0.5 + amount / 2)
    lab[0, ...] = l_ * 100.
    y = curves(x, 0.5 + amount / 2)
    j = torch.clamp(kornia.color.lab_to_rgb(lab), 0, 1)
    y = (2 * y + j) / 3
    return y[[2, 1, 0]]

def darken(x: torch.Tensor, amount: float, dolab: bool = False) -> torch.Tensor:
    x = x[[2, 1, 0], :, :]
    lab = kornia.color.rgb_to_lab(x)
    if dolab:
        l = lab[0, ...] / 100.
        l_ = curves(l, 0.5 + amount / 2)
        lab[0, ...] = l_ * 100.
    y = curves(x, 0.5 - amount / 2)
    if dolab:
        j = torch.clamp(kornia.color.lab_to_rgb(lab), 0, 1)
        y = (2 * y + j) / 3
    return y[[2, 1, 0]]

def jitter(x: torch.Tensor, amount: float) -> torch.Tensor:
    y = imscatter(x, amount, 5)
    return y

def quantization(x: torch.Tensor, levels: int) -> torch.Tensor:
    image = kornia.color.rgb_to_grayscale(x) * 255
    image = image.cpu().numpy()
    num_classes = levels
    hist, bins = np.histogram(image, num_classes, [0, 255])
    return_thresholds = np.zeros(num_classes - 1)
    for i in range(num_classes - 1): return_thresholds[i] = bins[i + 1]
    bins = torch.tensor([0] + return_thresholds.tolist() + [256])
    bins = bins.type(torch.int)
    image = torch.bucketize(x.contiguous() * 255., bins).to(torch.float32)
    image = mapmm(image)
    return image

def linear_contrast_change(x: torch.Tensor, amount: float) -> torch.Tensor:
    y = curves(x, [0.25 - amount / 4, 0.75 + amount / 4])
    return y

# --- PART 3: MAIN CALL ---
distortion_groups = {
    "blur": ["gausblur", "lensblur"],
    "color_distortion": ["colorshift", "colorsat"],
    "jpeg": ["jpeg"],
    "noise": ["whitenoise", "impulsenoise"],
    "brightness_change": ["brighten", "darken"],
    "spatial_distortion": ["jitter",  "quantization"],
    "sharpness_contrast": ["lincontrchange"],
}

distortion_range = {
    "gausblur": [0.1, 0.5, 1, 2, 5],
    "lensblur": [1, 2, 4, 6, 8],
    "colorshift": [1, 3, 6, 8, 12],
    "colorsat": [0.4, 0.2, 0.1, 0, -0.4],
    "jpeg": [43, 36, 24, 7, 4],
    "whitenoise": [0.001, 0.002, 0.003, 0.005, 0.01],
    "impulsenoise": [0.001, 0.005, 0.01, 0.02, 0.03],
    "brighten": [0.1, 0.2, 0.4, 0.7, 1.1],
    "darken": [0.05, 0.1, 0.2, 0.4, 0.8],
    "jitter": [0.05, 0.1, 0.2, 0.5, 1],
    "quantization": [20, 16, 13, 10, 7],
    "lincontrchange": [0., 0.15, -0.4, 0.3, -0.6],
}

distortion_functions = {
    "gausblur": gaussian_blur, "lensblur": lens_blur,
    "colorshift": color_shift, "colorsat": color_saturation,
    "jpeg": jpeg, "whitenoise": white_noise, "impulsenoise": impulse_noise,
    "brighten": brighten, "darken": darken,
    "jitter": jitter, "quantization": quantization,
    "lincontrchange": linear_contrast_change
}

def get_distortions_composition(max_distortions: int = 3, num_levels: int = 5):
    MEAN = 0; STD = 2.5
    num_distortions = random.randint(1, max_distortions)
    groups = random.sample(list(distortion_groups.keys()), num_distortions)
    distortions = [random.choice(distortion_groups[group]) for group in groups]
    distort_functions = [distortion_functions[dist] for dist in distortions]
    probabilities = [1 / (STD * np.sqrt(2 * np.pi)) * np.exp(-((i - MEAN) ** 2) / (2 * STD ** 2)) for i in range(num_levels)]
    normalized_probabilities = [prob / sum(probabilities) for prob in probabilities]
    distort_values = [np.random.choice(distortion_range[dist][:num_levels], p=normalized_probabilities) for dist in distortions]
    return distort_functions, distort_values

def distort_images(image: torch.Tensor, distort_functions: list = None, distort_values: list = None,
                   max_distortions: int = 3, num_levels: int = 5) -> torch.Tensor:
    image_loc = image.clone()
    if distort_functions is None or distort_values is None:
        distort_functions, distort_values = get_distortions_composition(max_distortions, num_levels)
    for distortion, value in zip(distort_functions, distort_values):
        image_loc = distortion(image_loc, value)
        image_loc = image_loc.to(torch.float32)
        image_loc = torch.clip(image_loc, 0, 1)
    return image_loc, distort_functions, distort_values

print("✅ Distortion Library Loaded (ALL Ops Enabled).")

In [ ]:
# Cell 3: Mathematical Preprocessors
def apply_pid(image_tensor, quantization_levels=8):
    """ Pixelwise Decomposition: RGB -> YUV -> Quantize -> RGB -> Absolute Difference """
    yuv_image = kornia.color.rgb_to_yuv(image_tensor.unsqueeze(0))
    yuv_quantized = torch.round(yuv_image * quantization_levels) / quantization_levels
    rgb_reconstructed = kornia.color.yuv_to_rgb(yuv_quantized).squeeze(0)
    residual_map = torch.abs(image_tensor - rgb_reconstructed)
    return residual_map

def apply_fft_magnitude(image_tensor):
    """ Frequency Domain: Grayscale -> 2D FFT -> Magnitude -> Log -> Normalize -> Float32 """
    gray = 0.299 * image_tensor[0] + 0.587 * image_tensor[1] + 0.114 * image_tensor[2]
    fft_complex = torch.fft.fft2(gray)
    fft_shifted = torch.fft.fftshift(fft_complex)
    
    magnitude = torch.abs(fft_shifted)
    magnitude_log = torch.log(magnitude + 1e-8)
    
    mag_min = magnitude_log.min()
    mag_max = magnitude_log.max()
    magnitude_normalized = (magnitude_log - mag_min) / (mag_max - mag_min + 1e-8)
    
    return magnitude_normalized.unsqueeze(0) # Returns (1, H, W)

In [ ]:
# Cell 4: Custom Dataset
class ContrastiveAIGenDataset(Dataset):
    def __init__(self, data_dir, mode='train', distortion_prob=0.7):
        self.mode = mode
        self.distortion_prob = distortion_prob
        self.image_paths = []
        self.labels = []
        
        # NOTE: Adapt this iteration to match your specific folder/CSV structure
        for class_label, class_name in enumerate(['real', 'fake']):
            class_dir = os.path.join(data_dir, class_name)
            if os.path.exists(class_dir):
                for img_name in os.listdir(class_dir):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(class_label)
                    
        print(f'Loaded {len(self.image_paths)} images for {mode}.')

        self.resize = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert("RGB")
        except:
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))

        image = self.resize(image)
        img_tensor = self.to_tensor(image)

        # Apply perturbations 70% of the time during training
        if self.mode == 'train' and random.random() < self.distortion_prob:
            try:
                img_tensor, _, _ = distort_images(img_tensor) # From Cell 2
            except:
                pass

        pid_residual = apply_pid(img_tensor)           
        fft_magnitude = apply_fft_magnitude(img_tensor) 

        # 4-Channel Fusion -> (4, 456, 456)
        four_channel_tensor = torch.cat([pid_residual, fft_magnitude], dim=0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return four_channel_tensor, label

In [ ]:
# Cell 5: Contrastive Model Architecture
class ContrastiveEffNetB5(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.backbone = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        
        # Modify Conv_Stem for 4 Channels
        original_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=4, 
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )
        
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = original_conv.weight
            new_conv.weight[:, 3:4, :, :] = torch.mean(original_conv.weight, dim=1, keepdim=True)
            
        self.backbone.features[0][0] = new_conv
        
        # Replace standard classification head with a Projection Head
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embedding_dim)
        )

    def forward(self, x):
        embeddings = self.backbone(x)
        embeddings = F.normalize(embeddings, p=2, dim=1) # L2 Normalize for Cosine Similarity
        return embeddings

In [ ]:
# Cell 6: Supervised Contrastive Loss
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = (features.device if features.is_cuda else torch.device('cpu'))
        batch_size = features.shape[0]
        
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        loss = - mean_log_prob_pos
        return loss.mean()

In [ ]:
# Cell 7: Checkpointing System
import os
import glob
import torch

def save_checkpoint(model, optimizer, epoch, best_loss):
    # Using epoch + 1 so the file matches the human-readable epoch number (e.g., Epoch 1)
    filename = f"effnet_b5_modified_{epoch+1}.pth"
    
    checkpoint = {
        'epoch': epoch, # Keep internal epoch 0-indexed for the loop
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_loss': best_loss
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved: {filename}")

def load_checkpoint(model, optimizer, filename=None):
    # If no specific filename is provided, auto-detect the latest one
    if filename is None:
        checkpoints = glob.glob("effnet_b5_modified_*.pth")
        if checkpoints:
            # Sort files by extracting the epoch number from the string
            checkpoints.sort(key=lambda x: int(x.split('_')[-1].split('.pth')[0]))
            filename = checkpoints[-1] # Grab the one with the highest number

    if filename and os.path.isfile(filename):
        print(f"Resuming from {filename}...")
        checkpoint = torch.load(filename, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        
        # Resume at the next epoch
        start_epoch = checkpoint['epoch'] + 1 
        best_loss = checkpoint.get('best_loss', float('inf'))
        print(f"Successfully loaded. Resuming at Epoch {start_epoch + 1}")
        return start_epoch, best_loss
    else:
        print("No checkpoint found. Starting fresh.")
        return 0, float('inf')

In [ ]:
# Cell 8: A100 Training Loop
def validate_contrastive(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            # non_blocking=True utilizes asynchronous DMA transfers
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            embeddings = model(images)
            loss = criterion(embeddings, labels)
            running_loss += loss.item()
    return running_loss / len(loader)

def train_contrastive():
    full_dataset = ContrastiveAIGenDataset(DATA_DIR, mode='train', distortion_prob=0.7)
    val_size = int(len(full_dataset) * VAL_SPLIT)
    train_size = len(full_dataset) - val_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
    
    # --- A100 DATALOADER OPTIMIZATIONS ---
    # num_workers=24 prevents the CPU from starving the A100
    # pin_memory=True allows ultra-fast data transfer
    # prefetch_factor=3 builds a queue of processed tensors
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=24, pin_memory=True, prefetch_factor=3, persistent_workers=True
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

    model = ContrastiveEffNetB5(embedding_dim=128).to(DEVICE)
    criterion = SupConLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda') 
    
    start_epoch, best_val_loss = load_checkpoint(model, optimizer)
    patience_counter = 0

    print(f"Starting Training on {DEVICE} (A100 Opt)...")
    
    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}")
        
        for images, labels in pbar:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            
            # set_to_none=True clears gradients completely, saving VRAM and slightly boosting speed
            optimizer.zero_grad(set_to_none=True) 
            
            with torch.amp.autocast('cuda'):
                embeddings = model(images)
                loss = criterion(embeddings, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

        # Validation & Early Stopping
        val_loss = validate_contrastive(model, val_loader, criterion)
        print(f"Epoch {epoch+1} | Train: {running_loss/len(train_loader):.4f} | Val: {val_loss:.4f}")
        
        # The function will now automatically name it 'effnet_b5_modified_X.pth'
        save_checkpoint(model, optimizer, epoch, best_val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_contrastive_model.pth")
            print(">>> New Best Model Saved!")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered.")
                break

# Uncomment to run training
if __name__ == "__main__":
    train_contrastive()

In [ ]:
# Cell 9: FIRE Verification Module
from diffusers import AutoencoderKL

class FIREVerifier(nn.Module):
    def __init__(self, device='cuda'):
        super().__init__()
        self.device = device
        print("Loading Stable Diffusion VAE for FIRE module...")
        # Uses standard SD VAE to reconstruct the image and check for anomalies
        self.vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
        self.vae.eval()
        
        # Freeze VAE weights
        for param in self.vae.parameters():
            param.requires_grad = False

    def get_mid_band_mask(self, shape, low_freq=0.1, high_freq=0.4):
        """ Creates a binary mask to isolate mid-frequency spectral anomalies """
        _, _, h, w = shape
        cy, cx = h // 2, w // 2
        Y, X = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
        dist = torch.sqrt((X - cx)**2 + (Y - cy)**2).to(self.device)
        max_dist = math.sqrt(cy**2 + cx**2)
        
        norm_dist = dist / max_dist
        mask = (norm_dist >= low_freq) & (norm_dist <= high_freq)
        return mask.float().unsqueeze(0).unsqueeze(0)

    @torch.no_grad()
    def evaluate(self, rgb_image_tensor):
        """ 
        1. Encodes image to latent space
        2. Decodes back to pixel space
        3. Measures mid-band frequency error between original and reconstruction
        """
        # VAE requires inputs bounded [-1, 1]
        x_in = rgb_image_tensor * 2.0 - 1.0 
        
        # Reconstruct
        latents = self.vae.encode(x_in).latent_dist.sample()
        reconstruction = self.vae.decode(latents).sample
        
        # Convert to Grayscale for FFT
        orig_gray = 0.299*x_in[:,0] + 0.587*x_in[:,1] + 0.114*x_in[:,2]
        recon_gray = 0.299*reconstruction[:,0] + 0.587*reconstruction[:,1] + 0.114*reconstruction[:,2]
        
        # Compute FFT
        fft_orig = torch.fft.fftshift(torch.fft.fft2(orig_gray))
        fft_recon = torch.fft.fftshift(torch.fft.fft2(recon_gray))
        
        # Extract Magnitudes
        mag_orig = torch.log(torch.abs(fft_orig) + 1e-8)
        mag_recon = torch.log(torch.abs(fft_recon) + 1e-8)
        
        # Apply Mid-Band Mask
        mask = self.get_mid_band_mask(mag_orig.unsqueeze(1).shape)
        masked_diff = torch.abs(mag_orig - mag_recon) * mask.squeeze(1)
        
        # Return scalar reconstruction error
        error_score = masked_diff.sum().item() / mask.sum().item()
        return error_score

In [ ]:
# Cell 10: Inference Pipeline (B5 Primary -> FIRE Tie-Breaker)
def run_two_stage_inference(image_path, b5_model, fire_module, centroids, margin_threshold=0.15):
    """
    centroids: Dict containing the computed average vector for 'real' and 'fake' classes
               from your training set. e.g., {'real': tensor, 'fake': tensor}
    """
    # 1. Load & Prepare
    raw_img = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor()
    ])
    img_tensor = transform(raw_img).to(DEVICE)
    
    # 2. Extract 4-Channel features for B5
    pid_res = apply_pid(img_tensor).to(DEVICE)
    fft_mag = apply_fft_magnitude(img_tensor).to(DEVICE)
    four_channel_in = torch.cat([pid_res, fft_mag], dim=0).unsqueeze(0)
    
    # 3. Stage 1: EfficientNet-B5 Prediction 
    b5_model.eval()
    with torch.no_grad():
        embedding = b5_model(four_channel_in)
        
    # Calculate Cosine Similarity to known clusters
    sim_real = F.cosine_similarity(embedding, centroids['real'].unsqueeze(0)).item()
    sim_fake = F.cosine_similarity(embedding, centroids['fake'].unsqueeze(0)).item()
    
    confidence_margin = abs(sim_real - sim_fake)
    
    # 4. Decision Router
    if confidence_margin > margin_threshold:
        # B5 is highly confident. Accept prediction.
        prediction = "Real" if sim_real > sim_fake else "AI-Generated"
        return {"prediction": prediction, "confidence": confidence_margin, "routed_to_fire": False}
    else:
        # 5. Stage 2: FIRE Verification (Tie-Breaker)
        # B5 is confused (image is on the boundary). Route original RGB to FIRE.
        print(f"Ambiguous prediction (Margin: {confidence_margin:.3f}). Routing to FIRE...")
        fire_error = fire_module.evaluate(img_tensor.unsqueeze(0))
        
        # High reconstruction error = Real Photograph (AI struggled to rebuild natural noise)
        # Low reconstruction error = AI-Generated (AI easily rebuilt AI artifacts)
        FIRE_THRESHOLD = 0.5 # You will need to calibrate this threshold on your validation set
        
        prediction = "Real" if fire_error > FIRE_THRESHOLD else "AI-Generated"
        return {"prediction": prediction, "fire_error_score": fire_error, "routed_to_fire": True}